# IAN 620 Final Project Starter Notebook

## Build and Evaluate a Retrieval-Augmented Generation (RAG) System

**Required stack**
- Dataset: SQuAD 1.1
- Baseline: TF-IDF + cosine similarity
- Embedding model: `sentence-transformers/all-MiniLM-L6-v2`
- Vector database: ChromaDB

This notebook is a starter structure, not a completed final project. Complete the marked student tasks, add your own evaluation, and explain your results in your own words.


## Project workflow

1. Load SQuAD.
2. Build a collection of unique context passages.
3. Explore the text.
4. Build a TF-IDF retrieval baseline.
5. Create MiniLM-L6 embeddings.
6. Store the embeddings in ChromaDB.
7. Retrieve the Top 3 passages for a question.
8. Compare TF-IDF with ChromaDB retrieval.
9. Build a RAG prompt.
10. Connect an LLM and evaluate the final answers.


## 0. Install the required libraries

Run this cell once if the libraries are not already installed.


In [ ]:
%pip install -q datasets pandas scikit-learn sentence-transformers chromadb numpy


## 1. Import the libraries


In [ ]:
import numpy as np
import pandas as pd
import chromadb

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


## 2. Load SQuAD 1.1

SQuAD gives us the three pieces we need:
- `context` = the passage we want to retrieve
- `question` = the user query
- `answers` = the known answer used during evaluation


In [ ]:
dataset = load_dataset("rajpurkar/squad")

train_df = pd.DataFrame(dataset["train"])
validation_df = pd.DataFrame(dataset["validation"])

print("Training examples:", len(train_df))
print("Validation examples:", len(validation_df))

validation_df.head()


### Student task
Briefly describe the dataset columns. Display two or three examples and explain how a question is connected to its context and answer.


## 3. Create the document collection

A context may appear in several rows because multiple questions can be written for the same passage. For retrieval, keep one copy of each context.


In [ ]:
documents_df = (
    validation_df[["title", "context"]]
    .drop_duplicates(subset="context")
    .reset_index(drop=True)
)

documents_df["doc_id"] = [
    f"doc_{i:04d}" for i in range(len(documents_df))
]

context_to_id = dict(
    zip(documents_df["context"], documents_df["doc_id"])
)

validation_df["correct_doc_id"] = validation_df["context"].map(context_to_id)

print("Unique context passages:", len(documents_df))
documents_df.head()


## 4. Explore the text

Keep this section simple and meaningful. You do not need a large EDA report.


In [ ]:
documents_df["word_count"] = documents_df["context"].str.split().str.len()

print(documents_df["word_count"].describe())

print("\nExample title:", documents_df.loc[0, "title"])
print("\nExample context:\n", documents_df.loc[0, "context"][:700])


### Student task
Add one or two useful observations about the corpus. Explain any cleaning decisions you make.


## 5. TF-IDF retrieval baseline

TF-IDF gives us a lexical baseline. It is often strong when the question and relevant passage share important words.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=20000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    documents_df["context"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)


In [ ]:
def retrieve_tfidf(question, top_k=3):
    question_vector = tfidf_vectorizer.transform([question])
    scores = cosine_similarity(question_vector, tfidf_matrix).flatten()

    top_indices = scores.argsort()[::-1][:top_k]

    results = documents_df.iloc[top_indices][
        ["doc_id", "title", "context"]
    ].copy()

    results["score"] = scores[top_indices]

    return results.reset_index(drop=True)


In [ ]:
sample_question = validation_df.loc[0, "question"]

print("Question:", sample_question)
retrieve_tfidf(sample_question, top_k=3)


### Student task
Test TF-IDF with several questions. Include one example where it works well and one where the result is less convincing. Explain why.


## 6. Load the required MiniLM-L6 embedding model

Use the same embedding model for documents and questions.


In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")


## 7. Create embeddings for the SQuAD contexts


In [ ]:
document_texts = documents_df["context"].tolist()

document_embeddings = embedding_model.encode(
    document_texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Embedding shape:", document_embeddings.shape)


## 8. Create the ChromaDB collection

We use a persistent Chroma client so the collection can remain available in the project folder.


In [ ]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_ian620_final"
)

collection = chroma_client.get_or_create_collection(
    name="squad_contexts"
)

print("Collection:", collection.name)


## 9. Store documents and MiniLM-L6 embeddings in ChromaDB

Using `upsert()` makes this cell safe to rerun without creating duplicate records.


In [ ]:
ids = documents_df["doc_id"].tolist()

metadatas = [
    {"title": title}
    for title in documents_df["title"].tolist()
]

BATCH_SIZE = 500

for start in range(0, len(documents_df), BATCH_SIZE):
    end = start + BATCH_SIZE

    collection.upsert(
        ids=ids[start:end],
        documents=document_texts[start:end],
        embeddings=document_embeddings[start:end].tolist(),
        metadatas=metadatas[start:end]
    )

print("Documents stored in ChromaDB:", collection.count())


## 10. Retrieve the Top 3 passages with ChromaDB

The question is embedded with the same MiniLM-L6 model, and ChromaDB searches for the nearest document vectors.


In [ ]:
def retrieve_chroma(question, top_k=3):
    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=question_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    rows = []

    for i in range(len(results["ids"][0])):
        rows.append({
            "doc_id": results["ids"][0][i],
            "title": results["metadatas"][0][i]["title"],
            "distance": results["distances"][0][i],
            "context": results["documents"][0][i]
        })

    return pd.DataFrame(rows)


In [ ]:
sample_question = validation_df.loc[0, "question"]

print("Question:", sample_question)
retrieve_chroma(sample_question, top_k=3)


### Student task
Run ChromaDB retrieval for several questions. For each example, check whether the correct SQuAD context appears in the Top 3 results.


## 11. Compare TF-IDF and ChromaDB retrieval


In [ ]:
question = validation_df.loc[10, "question"]

print("QUESTION")
print(question)

print("\nTF-IDF TOP 3")
display(retrieve_tfidf(question, top_k=3))

print("\nCHROMADB + MINILM-L6 TOP 3")
display(retrieve_chroma(question, top_k=3))


### Student task
Create a small comparison table for several questions. Discuss at least one case where semantic retrieval behaves differently from TF-IDF.


## 12. Build the RAG prompt

The retrieved passages become context for the LLM. Require the model to stay grounded in the retrieved text.


In [ ]:
def build_rag_prompt(question, retrieved_df):
    context_text = "\n\n".join(
        retrieved_df["context"].tolist()
    )

    prompt = f"""Use only the context below to answer the question.
If the context does not contain enough information, say that you do not have enough information.

Context:
{context_text}

Question:
{question}

Answer:
"""

    return prompt


In [ ]:
question = validation_df.loc[10, "question"]
retrieved = retrieve_chroma(question, top_k=3)
prompt = build_rag_prompt(question, retrieved)

print(prompt)


## 13. Connect your LLM

Use the LLM option approved for your project.

### Student task
Create a function named `generate_answer(prompt)` that sends the prompt to your selected LLM and returns the generated answer. Then test the question first without RAG context and again with RAG context.


In [ ]:
def generate_answer(prompt):
    # TODO: Connect the LLM you are using for the project.
    # Return only the model's answer as a string.
    raise NotImplementedError("Add your LLM connection here.")


## 14. Retrieval evaluation

For each validation question, SQuAD tells us which context is correct.

### Required metrics
- Hit@1: Is the correct context ranked first?
- Recall@3: Is the correct context anywhere in the Top 3?


In [ ]:
evaluation_sample = validation_df.sample(
    n=50,
    random_state=42
)[["question", "correct_doc_id", "answers"]].reset_index(drop=True)

evaluation_sample.head()


In [ ]:
def evaluate_retrieval(sample_df, retriever, top_k=3):
    # TODO:
    # 1. Loop through the questions.
    # 2. Retrieve top_k documents.
    # 3. Compare retrieved doc_id values with correct_doc_id.
    # 4. Calculate and return the retrieval success rate.
    pass


## 15. Final RAG evaluation

Create a results table with columns similar to:

| Question | Correct document retrieved? | Known answer | RAG answer | Supported by context? |
|---|---|---|---|---|

Evaluate multiple held-out questions. Include successful examples and at least one failure case.


## 16. Final discussion

Answer these questions in your own words:

1. How did TF-IDF retrieval compare with MiniLM-L6 + ChromaDB?
2. Did adding retrieved context improve the LLM answers? Give evidence.
3. When the system failed, was the main problem retrieval or generation?
4. What would you improve if you continued this project?
5. What did you learn about how RAG works end to end?
